In [11]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [12]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_15592/4028186587.py:2: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

/tmp/ipykernel_15592/4028186587.py:3: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [13]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'

In [14]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  


def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [15]:
n_points = 1750

def full_int(mg, a1, a2, m2_func, q2_val, sqrt_s):
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []

    for q2 in q2_val:
        def integrand(y, x, mg, a1, a2, m2_func, q2_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q2_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q2_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [16]:
eps_min = 0.0616
mg_min = 0.389
a1_min = 1.50	
a2_min = 2.13

In [17]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 1e-2
    max_q2   = 0.2001
    q2_step  = 0.001


    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(
                mg, a1, a2, mg_model, q2, sqrt_s
        )
        # print(integral_value)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale
        # print(f"q2 = {q2}, diff_t = {integral_value}, amp born = {amp_value:6e} \n")

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    eps_min,
    mg_min,
    a1_min,
    a2_min,
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [18]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [19]:
def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 8 * regge_factor * diff_T  



In [20]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad, dblquad
from functools import lru_cache

# ── Grid parameters ───────────────────────────────────────────────────────────
# Replace adaptive quad with fixed Gauss-Legendre grids.
# Tune N_B and N_Q for accuracy vs speed tradeoff.
N_B  = 256*2   # number of b  points in [0, b_max]
N_Q  = 256*2   # number of q  points in [0, q_max_chi]

q_max_chi = 1.0
q_max_eik = np.sqrt(0.1)
b_max     = 10.0

# ── Precompute Gauss-Legendre nodes & weights (once at import) ────────────────
# Nodes/weights on [-1,1], then mapped to [0, b_max] and [0, q_max_chi].
_gl_nodes, _gl_weights = np.polynomial.legendre.leggauss(N_B)

# b grid: [-1,1] → [0, b_max]
b_nodes   = 0.5 * b_max   * (_gl_nodes  + 1.0)          # shape (N_B,)
b_weights = 0.5 * b_max   *  _gl_weights                 # shape (N_B,)

# q grid: [-1,1] → [0, q_max_chi]
_gl_nodes_q, _gl_weights_q = np.polynomial.legendre.leggauss(N_Q)
q_nodes   = 0.5 * q_max_chi * (_gl_nodes_q + 1.0)       # shape (N_Q,)
q_weights = 0.5 * q_max_chi *  _gl_weights_q             # shape (N_Q,)


# ── Vectorized chi: computes chi for ALL b values at once ────────────────────
def chi_vec(b_arr, mg, a1, a2, eps, m2_func, sqrt_s):
    """
    Vectorized replacement for chi().

    Parameters
    ----------
    b_arr   : 1-D array of impact-parameter values, shape (N_B,)

    Returns
    -------
    chi_arr : complex array, shape (N_B,)

    Strategy
    --------
    chi(b) = (1/s) ∫₀^q_max  q · J₀(b·q) · A_born(q)  dq

    A_born depends only on q (not on b), so we compute it once on the
    q-grid and then evaluate the b-integral as a matrix–vector product.

    Shape broadcast:
        q         : (N_Q,)
        b_arr     : (N_B,)
        J₀(b·q)   : (N_B, N_Q)   (outer product)
        born(q)   : (N_Q,)        (complex)
        integrand : (N_B, N_Q)
        result    : (N_B,)        — one value per b
    """
    s_local = sqrt_s ** 2

    q2_arr   = q_nodes ** 2                              # (N_Q,)
    t_arr    = -q2_arr                                   # (N_Q,)

    # Evaluate A_born on the whole q-grid (vectorised over q)
    # full_int and amp_calculation must accept array inputs, OR we loop here.
    # We loop over q once (N_Q calls) — much cheaper than N_B × N_Q calls.
    born_arr = np.array([
        amp_calculation(
            full_int(mg, a1, a2, m2_func, q2, sqrt_s),
            s_local, eps, -q2
        )
        for q2 in q2_arr
    ], dtype=complex)                                    # (N_Q,)

    # Outer product of Bessel function: J₀(b · q)  →  (N_B, N_Q)
    bq       = np.outer(b_arr, q_nodes)                  # (N_B, N_Q)
    J0_mat   = j0(bq)                                    # (N_B, N_Q)

    # Integrand: q · J₀(b·q) · A_born(q) / s,  shape (N_B, N_Q)
    integrand_mat = (1.0 / s_local) * q_nodes * J0_mat * born_arr   # broadcasting

    # Gauss-Legendre quadrature: sum over q-axis weighted by q_weights
    chi_arr  = integrand_mat @ q_weights                 # (N_B,)  complex

    return chi_arr


# ── Vectorized model_function ─────────────────────────────────────────────────
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):
    """
    Fully vectorized over both b and q (outer) grids.

    The double integral
        T_eik(q_out) = i·s ∫₀^b_max  b · J₀(q_out·b) · [1 − e^{iχ(b)}]  db

    is evaluated as:
        1. chi_vec()  — vectorized over b_nodes  → chi_arr  shape (N_B,)
        2. kernel(b)  = b · [1 − exp(i·chi(b))]  → shape (N_B,)
        3. For each q_out in x: dot(kernel * J₀(q_out·b), b_weights)

    Step 3 is a matrix–vector product over all data points at once.
    """
    m2      = m2_log if model_type == 'log' else m2_pl
    s_local = sqrt_s ** 2

    # ── 1. Compute chi on the b-grid (single batch call) ─────────────────────
    chi_arr = chi_vec(b_nodes, mg, a1, a2, eps, m2, sqrt_s)   # (N_B,)  complex

    # ── 2. Compute the b-integrand kernel (independent of q_out) ─────────────
    kernel  = b_nodes * (1.0 - np.exp(-chi_arr))           # (N_B,)  complex

    # ── 3. Evaluate the outer integral for every q_out simultaneously ─────────
    # x contains q² values; q_out = sqrt(q²)
    q_out   = np.sqrt(np.asarray(x))                           # (N_x,)

    # J₀(q_out · b)  →  shape (N_x, N_B)
    J0_out  = j0(np.outer(q_out, b_nodes))                     # (N_x, N_B)

    # Weighted integrand: J₀(q_out·b) · kernel(b) · b_weight
    # Shape: (N_x, N_B) * (N_B,) * (N_B,)
    amp_eik = 1j * s_local * (J0_out * kernel * b_weights).sum(axis=1)  # (N_x,)

    # ── 4. Differential cross-section ─────────────────────────────────────────
    diff_sigma_eik = (
        (amp_eik.imag**2)
        * (np.pi / s_local**2)
        * 0.389379323
    )

    return diff_sigma_eik


# ── Energy-specific wrappers (unchanged interface) ───────────────────────────
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='log')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='log')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='log')


# ── Minuit setup (unchanged) ──────────────────────────────────────────────────
chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7,  verbose=True)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8,  verbose=True)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13, verbose=True)

chi2_total = chi2_7 

eps_min = 0.094763
mg_min  = 0.93
a1_min  = 1.4
a2_min  = 2.69

minuit_eik = Minuit(
    chi2_total,
    mg  = mg_min,
    a1  = a1_min,
    a2  = a2_min,
    eps = eps_min
)

minuit_eik.limits['mg'] = (0.3, 1.0)
minuit_eik.limits['a1'] = (0.1, 3.0)
minuit_eik.limits['a2'] = (0.1, 3.0)
minuit_eik.limits['eps'] = (0.0, 0.5)

minuit_eik.migrad()

minuit_eik.hesse()


# ─────────────────────────────────────────────────────────────────────────────
# ACCURACY CHECK — run this before fitting to validate the grid resolution.
# Compare vectorized output against the original quad-based result on one point.
# ─────────────────────────────────────────────────────────────────────────────
# from original import chi as chi_orig, model_function as model_orig
#
# q2_test = np.array([0.01])
# ref  = model_orig(q2_test, eps_min, mg_min, a1_min, a2_min, sqrt_s=7000)
# fast = model_7(q2_test,    eps_min, mg_min, a1_min, a2_min)
# print(f"ref={ref[0]:.6e}  fast={fast[0]:.6e}  rel_err={abs(fast[0]-ref[0])/ref[0]:.2e}")
#
# Increase N_B / N_Q if rel_err is too large (> 1e-3).
# ─────────────────────────────────────────────────────────────────────────────

(np.float64(0.094763), np.float64(0.93), np.float64(1.4), np.float64(2.69)) -> 5292.174576577027
(np.float64(0.09479666676480794), np.float64(0.93), np.float64(1.4), np.float64(2.69)) -> 5281.733023374568
(np.float64(0.09472933781651623), np.float64(0.93), np.float64(1.4), np.float64(2.69)) -> 5302.618727618004
(np.float64(0.094763), np.float64(0.9303307191994876), np.float64(1.4), np.float64(2.69)) -> 5310.023639455493
(np.float64(0.094763), np.float64(0.9296685848923966), np.float64(1.4), np.float64(2.69)) -> 5274.292790305513
(np.float64(0.094763), np.float64(0.9300919186137536), np.float64(1.4), np.float64(2.69)) -> 5297.134986004839
(np.float64(0.094763), np.float64(0.9299080277102687), np.float64(1.4), np.float64(2.69)) -> 5287.211643155894
(np.float64(0.094763), np.float64(0.93), np.float64(1.400497360182963), np.float64(2.69)) -> 5292.969122397148
(np.float64(0.094763), np.float64(0.93), np.float64(1.399502657655375), np.float64(2.69)) -> 5291.380144335228
(np.float64(0.094763)

KeyboardInterrupt: 